# Verify Agent Sign-In with PyJWT

## What you will build

Northwind Payroll runs an HR assistant agent. When an employee asks for a payslip, the agent calls
the payroll service for that employee, and it carries a signed token that says who is asking. A token
is a signed string, not the unit a model reads and bills in.

Without a check, the payroll service believes any signed token it receives. A token made for the
payslip service can then open the bank-details service, and an expired token still gets an answer.
The diagram shows the three mistakes this course stops:

- A token meant for payslips opens the bank-details service.
- An expired token still gets an answer.
- Any signed token is believed.

![What you will build](images/sign-in-desk-overview.svg)

## Step 0: Set up PyJWT and the Northwind test tokens

We import PyJWT, fix the secret and the clock, and build the tokens the tests use. Every token is
built inside this notebook, so the same code gives the same answers on every run.

The secret is a fake demo string, and the clock is a fixed number rather than the real time. PyJWT
normally reads the real clock, so this lesson switches those checks off and compares expiry with the
fixed number itself.

In [ ]:
import base64
import json

import jwt

SECRET = "northwind-demo-secret-for-teaching-only"
ATTACKER_SECRET = "attacker-guessed-secret-not-northwind-0001"
NOW = 1_800_000_000
ISSUER = "https://sign-in.northwind.example"
AUDIENCE = "payslip-service"
NEEDED_SCOPE = "payslip:read"
CLOCK_CHECKS_OFF = {"verify_exp": False, "verify_iat": False, "verify_nbf": False}

print(f"PyJWT {jwt.__version__} is ready. The clock is fixed at {NOW}.")

Every agent token carries the same claims. The function below builds them, and each keyword
argument replaces one claim. A value of `None` removes that claim, so one call can build each bad case.

In [ ]:
def make_agent_claims(**overrides):
    claims = {
        "iss": ISSUER,
        "aud": AUDIENCE,
        "sub": "emp-1042",
        "act": {"sub": "agent-hr-assistant"},
        "scope": "payslip:read",
        "iat": NOW,
        "exp": NOW + 300,
    }
    claims.update(overrides)
    return {name: value for name, value in claims.items() if value is not None}


def make_agent_token(secret=SECRET, **overrides):
    return jwt.encode(make_agent_claims(**overrides), secret, algorithm="HS256")


good_token = make_agent_token()
print(good_token[:48] + "...")
print(json.dumps(jwt.decode(good_token, options={"verify_signature": False}), indent=2))

Some bad tokens are signed with the wrong key. Others are edited after signing, expired, or aimed at
the wrong service. The helpers below build each case, and each case says which answer it should get.

In [ ]:
def encode_base64url(part):
    raw = json.dumps(part).encode()
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode()


def make_unsigned_token(claims):
    header = encode_base64url({"alg": "none", "typ": "JWT"})
    return f"{header}.{encode_base64url(claims)}."


def make_tampered_token(token, **new_claims):
    header, payload, signature = token.split(".")
    padded = payload + "=" * (-len(payload) % 4)
    claims = json.loads(base64.urlsafe_b64decode(padded))
    claims.update(new_claims)
    return f"{header}.{encode_base64url(claims)}.{signature}"


print(make_unsigned_token(make_agent_claims())[:48] + "...")

Now we list every case with the answer it should get. The last field is `True` for a good token,
which must be allowed, and `False` for a bad token, which must be refused.

In [ ]:
def build_test_cases():
    return [
        ("good: plain valid", make_agent_token(), True),
        ("good: two scopes", make_agent_token(scope="payslip:read payslip:list"), True),
        ("good: expires in one second", make_agent_token(exp=NOW + 1), True),
        ("good: second employee", make_agent_token(sub="emp-2001"), True),
        ("bad: expired", make_agent_token(exp=NOW - 60), False),
        ("bad: wrong audience", make_agent_token(aud="bank-details-service"), False),
        ("bad: wrong issuer", make_agent_token(iss="https://lookalike.example"), False),
        ("bad: signed with another secret", make_agent_token(secret=ATTACKER_SECRET), False),
        ("bad: payload edited after signing",
         make_tampered_token(make_agent_token(), scope="payslip:read bank:write"), False),
        ("bad: alg none, no signature", make_unsigned_token(make_agent_claims()), False),
        ("bad: missing act claim", make_agent_token(act=None), False),
        ("bad: scope is bank:write only", make_agent_token(scope="bank:write"), False),
    ]


CASES = build_test_cases()
good_count = sum(1 for _, _, should_allow in CASES if should_allow)
print(f"good tokens: {good_count}, bad tokens: {len(CASES) - good_count}")

## Step 1: Show that a signed token is not a trusted token

We start with the starter, which reads each token and trusts whatever it says. A **token** is the
signed string the agent sends with each call. It is not the unit a model reads and bills in.

A **claim** is one statement inside a token, such as which employee is asking. Reading a claim needs
no key, so a claim is only a statement until a check proves it.

In [ ]:
def trust_any_signed_token(token, needed_scope):
    try:
        jwt.decode(token, options={"verify_signature": False})
    except jwt.InvalidTokenError as error:
        return False, f"unreadable: {type(error).__name__}"
    return True, "readable, so trusted"


print(trust_any_signed_token(CASES[4][1], NEEDED_SCOPE))

To score a whole run, we need a way to send every case through one evaluator and count the mistakes.
The helpers below do that, and each evaluator returns a pair: whether the call is allowed, and why.

In [ ]:
def run_cases_through(evaluator):
    rows = []
    for name, token, should_allow in CASES:
        allowed, reason = evaluator(token, NEEDED_SCOPE)
        rows.append((name, should_allow, allowed, reason))
    return rows


def count_mistakes(rows):
    forbidden_accepted = sum(1 for _, should, allowed, _ in rows if allowed and not should)
    valid_rejected = sum(1 for _, should, allowed, _ in rows if should and not allowed)
    return forbidden_accepted, valid_rejected


def print_case_table(rows):
    for name, _, allowed, reason in rows:
        verdict = "allowed" if allowed else "refused"
        print(f"{name:<34} {verdict:<8} {reason}")


print("helpers ready")

Now we run every case through the starter and read the table and the two numbers.

In [ ]:
starter_rows = run_cases_through(trust_any_signed_token)
print_case_table(starter_rows)
forbidden_accepted, valid_rejected = count_mistakes(starter_rows)
print(f"starter forbidden_accepted: {forbidden_accepted}")
print(f"starter valid_rejected: {valid_rejected}")

`forbidden_accepted` counts the bad tokens let through, and `valid_rejected` counts the good tokens
refused. The starter lets through every bad token and refuses no good token, so its first number is 8.

## Step 2: Check the signature and pin the algorithm

Now we check the signature against the service's own secret, so a forged token fails. A token's
header can name its own algorithm, and the name "none" means the token has no signature at all.

**Algorithm pinning** means the code lists the one algorithm it accepts, so a header cannot switch
the check off. The expiry and audience checks stay off for now, because the next step adds them back.

![Check the signature and pin the algorithm](images/sign-in-checks-step-1.svg)

In [ ]:
def check_signature_only(token, needed_scope):
    try:
        jwt.decode(token, SECRET, algorithms=["HS256"],
                   options={**CLOCK_CHECKS_OFF, "verify_aud": False})
    except jwt.InvalidTokenError as error:
        return False, f"{type(error).__name__}"
    return True, "signature matches"


print_case_table(run_cases_through(check_signature_only))

The three tokens with a broken signature are refused now, and that includes the alg none token. The
expired, wrong-audience, wrong-issuer, missing-act and bank-only tokens still get through. The next
steps refuse them.

## Step 3: Check who the token is for and when it expires

Now we check the audience, the issuer and the expiry time. The **audience** is the service a token was
made for, so a payslip token must name the payslip service. The **expiry** is the exp claim, the
moment after which a token must be refused.

![Check who the token is for and when it expires](images/sign-in-checks-step-2.svg)

In [ ]:
def read_claims_for_service(token):
    return jwt.decode(
        token, SECRET, algorithms=["HS256"], audience=AUDIENCE, issuer=ISSUER,
        options={**CLOCK_CHECKS_OFF, "require": ["exp", "aud", "iss"]},
    )


def check_not_expired(claims):
    return claims["exp"] > NOW


def check_audience_issuer_and_expiry(token, needed_scope):
    try:
        claims = read_claims_for_service(token)
    except jwt.InvalidTokenError as error:
        return False, f"{type(error).__name__}"
    if not check_not_expired(claims):
        return False, "expired: exp is not after NOW"
    return True, "audience, issuer and expiry pass"


print_case_table(run_cases_through(check_audience_issuer_and_expiry))

The expired token, the wrong audience and the wrong issuer are refused now. The missing act claim and
the bank-only scope still get through, which the next two steps handle.

## Step 4: Check the agent acts for a named employee

Next we check delegation, so that every call is tied to a named employee. A call made on behalf of an employee needs two facts: the sub claim names
the employee, and the act claim names the agent that is acting. The **delegation** check refuses a
token with no act claim, because nothing then shows who is calling.

![Check the agent acts for a named employee](images/sign-in-checks-step-3.svg)

In [ ]:
def check_delegation(claims):
    act = claims.get("act")
    return bool(claims.get("sub")) and isinstance(act, dict) and bool(act.get("sub"))


def check_agent_is_delegated(token, needed_scope):
    allowed, reason = check_audience_issuer_and_expiry(token, needed_scope)
    if not allowed:
        return allowed, reason
    if not check_delegation(read_claims_for_service(token)):
        return False, "no agent acting for a named employee"
    return True, "agent acts for a named employee"


print_case_table(run_cases_through(check_agent_is_delegated))

The missing act claim is refused now. Only the bank-only scope token still gets through, and the scope
check in the next step refuses it.

## Step 5: Allow only the scope the call needs

Last of the checks, we read the scope claim. The **scope** lists what the token may do, as words
separated by spaces. The payslip call needs payslip:read, so a token that grants only bank:write must
be refused.

![Allow only the scope the call needs](images/sign-in-checks-step-4.svg)

In [ ]:
def check_scope(claims, needed_scope):
    return needed_scope in claims.get("scope", "").split()


def check_scope_for_call(token, needed_scope):
    allowed, reason = check_agent_is_delegated(token, needed_scope)
    if not allowed:
        return allowed, reason
    if not check_scope(read_claims_for_service(token), needed_scope):
        return False, "scope does not cover this call"
    return True, "scope covers this call"


print_case_table(run_cases_through(check_scope_for_call))

The bank-only scope token is refused now, so every bad token fails for a stated reason.

## Step 6: Combine the checks into one evaluator

Now we put every check in one function, so a reader sees the whole rule in one place. The decode call
checks the signature, the audience and the issuer, and the function then runs the claim checks in
order. It stops at the first refusal.

In [ ]:
def evaluate_agent_token(token, needed_scope):
    try:
        claims = read_claims_for_service(token)
    except jwt.InvalidTokenError as error:
        return False, f"{type(error).__name__}"
    if not check_not_expired(claims):
        return False, "expired: exp is not after NOW"
    if not check_delegation(claims):
        return False, "no agent acting for a named employee"
    if not check_scope(claims, needed_scope):
        return False, "scope does not cover this call"
    return True, "all checks pass"


print_case_table(run_cases_through(evaluate_agent_token))

Every good token is allowed, and every bad token is refused with a reason. The two numbers below sum
that up, and both should be zero.

In [ ]:
final_rows = run_cases_through(evaluate_agent_token)
forbidden_accepted, valid_rejected = count_mistakes(final_rows)
print(f"forbidden_accepted: {forbidden_accepted}")
print(f"valid_rejected: {valid_rejected}")

## Step 7: Test the evaluator without any key

Last we turn each rule into a test. Each test builds its tokens in this notebook, so it needs no key
and no network. The final test asserts the two numbers over every case, and the first failing test
stops the run.

In [ ]:
def is_allowed(evaluator, token):
    allowed, _ = evaluator(token, NEEDED_SCOPE)
    return allowed


def test_signature_and_algorithm_pinning():
    assert is_allowed(check_signature_only, make_agent_token())
    assert not is_allowed(check_signature_only, make_agent_token(secret=ATTACKER_SECRET))
    assert not is_allowed(check_signature_only, make_unsigned_token(make_agent_claims()))


def test_audience_issuer_and_expiry():
    assert is_allowed(check_audience_issuer_and_expiry, make_agent_token(exp=NOW + 1))
    assert not is_allowed(check_audience_issuer_and_expiry, make_agent_token(aud="bank-details-service"))
    assert not is_allowed(check_audience_issuer_and_expiry, make_agent_token(iss="https://lookalike.example"))
    assert not is_allowed(check_audience_issuer_and_expiry, make_agent_token(exp=NOW - 60))


print(is_allowed(check_signature_only, make_agent_token()))

The second group covers delegation and scope, and its last test checks the two numbers.

In [ ]:
def test_delegation_needs_an_agent_and_an_employee():
    assert is_allowed(check_agent_is_delegated, make_agent_token())
    assert not is_allowed(check_agent_is_delegated, make_agent_token(act=None))
    assert not is_allowed(check_agent_is_delegated, make_agent_token(sub=None))


def test_scope_must_cover_the_call():
    assert is_allowed(check_scope_for_call, make_agent_token(scope="payslip:read payslip:list"))
    assert not is_allowed(check_scope_for_call, make_agent_token(scope="bank:write"))


def test_evaluator_meets_the_case_metric():
    forbidden_accepted, valid_rejected = count_mistakes(run_cases_through(evaluate_agent_token))
    assert forbidden_accepted == 0, f"forbidden_accepted: {forbidden_accepted}"
    assert valid_rejected == 0, f"valid_rejected: {valid_rejected}"


TESTS = [
    test_signature_and_algorithm_pinning,
    test_audience_issuer_and_expiry,
    test_delegation_needs_an_agent_and_an_employee,
    test_scope_must_cover_the_call,
    test_evaluator_meets_the_case_metric,
]
for test in TESTS:
    test()
    print(f"passed: {test.__name__}")

## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Signature check** | `check_signature_only` | Proves a key signed the token and that nobody edited the claims after signing |
| **Algorithm pinning** | `algorithms=["HS256"]` | Accepts only the one algorithm the code names, so `none` is refused |
| **Audience** | `read_claims_for_service` | Refuses a token that was made for another service |
| **Expiry** | `check_not_expired` | Refuses a token whose exp is not after the fixed clock |
| **Delegation** | `check_delegation` | Requires a sub for the employee and an act claim for the agent |
| **Scope** | `check_scope` | Requires the one permission this call needs |